# 06 - Model Comparison & Selection

**Goal:** Systematic comparison of all models. Pick the winner. Save for production use.

In [ ]:
import os
import sys
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.insert(0, PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'prediction'))

from prediction.src.evaluation import evaluate_model, plot_predictions_vs_actual

DATA_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'data')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'prediction', 'models')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 100

df = pd.read_parquet(os.path.join(DATA_DIR, 'features_all.parquet'))
with open(os.path.join(DATA_DIR, 'feature_columns.json')) as f:
    FEATURE_COLS = json.load(f)
with open(os.path.join(DATA_DIR, 'best_xgb_params.json')) as f:
    best_xgb_params = json.load(f)

TARGET = 'total_points'

max_gw = df['round'].max()
split_gw = int(max_gw * 0.75)
train = df[df['round'] <= split_gw]
test = df[df['round'] > split_gw]
X_train, y_train = train[FEATURE_COLS], train[TARGET]
X_test, y_test = test[FEATURE_COLS], test[TARGET]

print(f'Train: {len(train):,} rows | Test: {len(test):,} rows')

## 1. Re-train All Models for Fair Comparison

Re-train every model here so they're all on the same train/test split.

In [ ]:
models = {}
predictions = {}
results = {}

# 1. Predict Mean
mean_pred = np.full(len(y_test), y_train.mean())
results['Predict Mean'] = evaluate_model(y_test, mean_pred)
predictions['Predict Mean'] = mean_pred

# 2. Recent Form (3GW rolling)
form_pred = test['pts_rolling_3'].values
results['Recent Form (3GW)'] = evaluate_model(y_test, form_pred)
predictions['Recent Form (3GW)'] = form_pred

# 3. Linear Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
lr = LinearRegression().fit(X_train_scaled, y_train)
lr_pred = lr.predict(X_test_scaled)
results['Linear Regression'] = evaluate_model(y_test, lr_pred)
predictions['Linear Regression'] = lr_pred
models['Linear Regression'] = (lr, scaler)

# 4. Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=10,
                           random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_pred = rf.predict(X_test)
results['Random Forest'] = evaluate_model(y_test, rf_pred)
predictions['Random Forest'] = rf_pred
models['Random Forest'] = rf

# 5. XGBoost (tuned)
xgb = XGBRegressor(**best_xgb_params, random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)
results['XGBoost (tuned)'] = evaluate_model(y_test, xgb_pred)
predictions['XGBoost (tuned)'] = xgb_pred
models['XGBoost (tuned)'] = xgb

print('All models re-trained.')

## 2. Summary Table

In [ ]:
results_df = pd.DataFrame(results).T.sort_values('MAE')
results_df.index.name = 'Model'

print('Model Comparison (sorted by MAE — lower is better):')
print('=' * 55)
print(results_df.round(3).to_string())

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for i, metric in enumerate(['MAE', 'RMSE', 'R2']):
    vals = results_df[metric]
    colors = ['#2ecc71' if v == vals.min() else '#3498db' for v in vals] if metric != 'R2' else \
             ['#2ecc71' if v == vals.max() else '#3498db' for v in vals]
    axes[i].barh(results_df.index, vals, color=colors)
    axes[i].set_xlabel(metric)
    axes[i].set_title(f'{metric} by Model')

plt.tight_layout()
plt.show()

## 3. Per-Position MAE Breakdown

In [ ]:
pos_results = []

for pos in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    mask = test['Position'] == pos
    if mask.sum() == 0:
        continue
    row = {'Position': pos, 'n': mask.sum()}
    for model_name, pred in predictions.items():
        row[model_name] = evaluate_model(y_test[mask], pred[mask])['MAE']
    pos_results.append(row)

pos_df = pd.DataFrame(pos_results).set_index('Position')
print('MAE by Position and Model:')
print(pos_df.round(3).to_string())

# Plot
model_cols = [c for c in pos_df.columns if c != 'n']
pos_df[model_cols].plot(kind='bar', figsize=(12, 5), alpha=0.8)
plt.ylabel('MAE')
plt.title('MAE by Position and Model')
plt.xticks(rotation=0)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 4. Top-Performer Accuracy

For FPL, the real question is: **can the model identify who will score the most points?**

For each test GW, we compare the model's top-20 predicted performers with the actual top-20.

In [ ]:
def top_k_overlap(test_df, predictions_array, k=20):
    """For each GW, compute overlap between top-k predicted and top-k actual."""
    test_copy = test_df[['element', 'round', TARGET]].copy()
    test_copy['predicted'] = predictions_array
    
    overlaps = []
    for gw in test_copy['round'].unique():
        gw_data = test_copy[test_copy['round'] == gw]
        if len(gw_data) < k:
            continue
        top_actual = set(gw_data.nlargest(k, TARGET)['element'])
        top_predicted = set(gw_data.nlargest(k, 'predicted')['element'])
        overlap = len(top_actual & top_predicted) / k
        overlaps.append(overlap)
    
    return np.mean(overlaps) if overlaps else 0

print(f'{"Model":<25s} {"Top-20 Overlap":>15s}')
print('-' * 42)
for model_name, pred in predictions.items():
    overlap = top_k_overlap(test, pred, k=20)
    print(f'{model_name:<25s} {overlap:15.1%}')

## 5. Error Analysis: Worst Predictions

When does the model fail badly? Understanding failure modes helps improve features.

In [ ]:
# Use the best model's predictions
best_model_name = results_df.index[0]  # Lowest MAE
best_pred = predictions[best_model_name]

test_analysis = test[['element', 'round', TARGET, 'Position']].copy()
test_analysis['predicted'] = best_pred
test_analysis['error'] = test_analysis[TARGET] - test_analysis['predicted']
test_analysis['abs_error'] = test_analysis['error'].abs()

# Merge player names
from sqlalchemy import create_engine
sys.path.insert(0, PROJECT_ROOT)
from src.pg_config import get_pg_config
cfg = get_pg_config()
engine = create_engine(f'postgresql+psycopg2://{cfg.user}:{cfg.password}@{cfg.host}:{cfg.port}/{cfg.dbname}')
players = pd.read_sql('SELECT "ID" as element, "Second Name" as name FROM gold.players', engine)
test_analysis = test_analysis.merge(players, on='element', how='left')

print(f'\nTop 15 WORST predictions (biggest absolute error) — model: {best_model_name}:')
worst = test_analysis.nlargest(15, 'abs_error')[['name', 'round', 'Position', TARGET, 'predicted', 'error']]
worst['predicted'] = worst['predicted'].round(1)
worst['error'] = worst['error'].round(1)
print(worst.to_string(index=False))

print('\nCommon causes of big errors: hauls from penalties, red cards, injury subs, own goals.')

## 6. Save Best Model

In [ ]:
# Retrain best model on ALL data (train + test) for production use
print(f'Best model: {best_model_name}')
print(f'Retraining on all {len(df):,} rows for production use...\n')

if 'XGBoost' in best_model_name:
    final_model = XGBRegressor(**best_xgb_params, random_state=42, n_jobs=-1)
    final_model.fit(df[FEATURE_COLS], df[TARGET])
    model_type = 'xgboost'
elif 'Random Forest' in best_model_name:
    final_model = RandomForestRegressor(n_estimators=100, max_depth=15, min_samples_leaf=10,
                                        random_state=42, n_jobs=-1)
    final_model.fit(df[FEATURE_COLS], df[TARGET])
    model_type = 'random_forest'
else:
    scaler_full = StandardScaler()
    X_all_scaled = scaler_full.fit_transform(df[FEATURE_COLS])
    final_model = LinearRegression().fit(X_all_scaled, df[TARGET])
    model_type = 'linear_regression'
    # Save scaler alongside model
    joblib.dump(scaler_full, os.path.join(MODELS_DIR, 'scaler.joblib'))

# Save model
model_path = os.path.join(MODELS_DIR, 'best_model.joblib')
joblib.dump(final_model, model_path)

# Save metadata
metadata = {
    'model_type': model_type,
    'model_name': best_model_name,
    'feature_columns': FEATURE_COLS,
    'target': TARGET,
    'train_rows': len(df),
    'test_metrics': results[best_model_name],
    'best_params': best_xgb_params if 'XGBoost' in best_model_name else None,
}
with open(os.path.join(MODELS_DIR, 'model_metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Model saved to: {model_path}')
print(f'Metadata saved to: {os.path.join(MODELS_DIR, "model_metadata.json")}')
print(f'\nModel type: {model_type}')
print(f'Test metrics: {results[best_model_name]}')

## 7. Final Results

In [ ]:
results_df.to_csv(os.path.join(DATA_DIR, 'results_final.csv'))

print('Final Model Comparison:')
print('=' * 55)
print(results_df.round(3).to_string())
print(f'\nBest model: {best_model_name}')
print(f'Saved to: prediction/models/best_model.joblib')

## Summary

**Model selection complete.**

Artifacts saved:
- `prediction/models/best_model.joblib` — the trained model
- `prediction/models/model_metadata.json` — model type, features, metrics
- `prediction/data/feature_columns.json` — feature list (needed for inference)
- `prediction/data/results_final.csv` — all model comparisons

Next: Phase 7 — `predict.py` (inference pipeline) and Streamlit integration.